# FR Fig. 09 - Weekly Operation

Service: `FR`  
BESS solution: `bus2017`, `2000 kWh`, `500 kW`  
Simulation window: hours `4657 to 4824`  
Output folder: `FR_Fig_09_Weekly_Operation`


In [1]:
e_e_hora_f = 167
e_hora_f = e_e_hora_f + 1

import sys
import os
import random
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time

sys.path.append(os.getcwd())

script_dir = os.getcwd()
print(f"script_dir: {script_dir}")

from f_PJM_buses import f_PJM_buses
from f_BESS_Sv_FR import f_BESS_Sv_FR
from f_BESS_Sv_PS import f_BESS_Sv_PS
from f_BESS_Sv_STK import f_BESS_Sv_STK

from datetime import datetime

# Configurar pandas para mostrar todas las filas y columnas sin cortes
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

############################################################
# ======= CODE START TIME =============
############################################################

print(f"\n📌 Start Time\t: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

######################################################################
# ==================== PJM SUITABLE BUSES ============================
######################################################################

bus_data_df, suitable_buses_df = f_PJM_buses()
df_length = len(suitable_buses_df)
# print(f"suitable_buses_df:\n{suitable_buses_df}\n")


######################################################################
# ==================== B E S S    T E S T   ==========================
######################################################################

# -------------------- BESS Service

Svc                  = 'FR'

# -------------------- BESS CONFIGURATIONS

n_dc                 = 0.9              # eficiencia de descarga
n_ch                 = 0.9              # eficiencia de carga
primera_hora         = 4657             # 4657 → 14 de julio pero corresponde a lo obtenido durante la hora 4657 por lo que el SoC_i debe serlo considerado en la hora anterior
ultima_hora          = 4824             # 20 de julio

# 46559  0.119061  4655            4 2018-07-13 23:54:00    23      54   13      7
# 46560  0.129343  4656            4 2018-07-14 00:00:00     0       0   14      7

b                    = 0
SoC_i                = 0.119061         # SoC_i 13 de julio (ultimo time step)

# -------------------- Parámetros Financieros

USD_kW               = 350             #145 #245                 # USD/kW
USD_kWh              = 350              #45 #345                # USD/kWh
OPEX_USD_kW_year     = 0.025 * USD_kW
annual_discount_rate = 0.02             # 2%
years                = 1000             # Años referenciados


######################################################################
# ================== EJECUCIÓN DE LA FUNCION  ========================
######################################################################

results = []

# Tipo de Sistema	   Potencia (kW)	   Capacidad de Almacenamiento (kWh)
# Mediana escala	   100 kW -500 kW	   200 kWh - 2,000 kWh

# -------------------- CONFIGURCIONES DE FUERZA BRUTA

#bus_indices = range(df_length)
bus_indices = range(24, 25, 1)           # bus2017
P_BESS_values = range(500, 501, 100)
E_BESS_values = range(2000, 2001, 100)

# ¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨

# -------------------- CONFIGURCIONES DE FUERZA BRUTA
# #bus3076
# bus_indices = range(99, 100, 1)  #range(df_length)
# # bus_indices = [99]
# P_BESS_values = range(100, 401, 100)
# E_BESS_values = range(1000, 2001, 200)

# ¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨

total_iterations = len(bus_indices) * len(P_BESS_values) * len(E_BESS_values)
current_iteration = 1

print("##" * 40) 
print(f" *************** INFORMATION ABOUT SUITABLE ITERATIONS *************** ")
print(f"Num. BatteryNodes: {len(bus_indices)}")
print(f"Num. P_BESS: {len(P_BESS_values)}")
print(f"Num. E_BESS: {len(E_BESS_values)}")
print(" - - " * 40) 
print(f"Total Iterations: {total_iterations}")
print(f"Hours: {primera_hora} to {ultima_hora}")
print("##" * 40) 
print("\n")

# ¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨¨

# BESS_cases = [
#     (100, 1000),  # Caso original
#     (500, 3000),  # Caso original
#     (400, 2000),  # Caso original
#     # (300, 2000),  # Caso original
#     # (200, 3000), # Caso original
#     # (400, 3000),  # Nuevo caso
# ]

# ================== ITERACION SOBRE LA FUERZA BRUTA
for index in bus_indices:
    row = suitable_buses_df.iloc[index]
    BatteryNode = row['bus']
    phases = row['num_phases']
    base_kv_line = row['base_kv_line']

    for P_BESS in P_BESS_values:
        for E_BESS in E_BESS_values:
            
            # Comparar solo con las combinaciones exactas de BESS_cases
            # if (P_BESS, E_BESS) in BESS_cases:

                print(f" ------------- BESS CASE: {P_BESS} kW, {E_BESS} kWh ------------- ")
                start_time = time.time()  # Iniciar el cronómetro
                print("-" * 80) 
                print(f"Iteration {current_iteration}/{total_iterations}")

                # Perfil de Potencia de BESS
                df_info_STK, annual_revenue, f1, f2 ,f3, f4, f5, f6, f_i_1, f_i_2, f_i_3 = f_BESS_Sv_FR(P_BESS, E_BESS, n_dc, n_ch, BatteryNode, primera_hora, ultima_hora, annual_discount_rate, SoC_i) 
                #print(f"df_info_STK: \n{df_info_STK}\n")
                end_time = time.time()  # Detener el cronómetro
                elapsed_time = (end_time - start_time) / 60   # Calcular el tiempo transcurrido

                # Guardar los resultados en la lista
                results.append({
                    'BatteryNode': BatteryNode,
                    'P_BESS': P_BESS,
                    'E_BESS': E_BESS,
                    'Annual_Revenue': annual_revenue,
                    'f1': f1,
                    'f2': f2,
                    'f3': f3,
                    'f4': f4,
                    'f5': f5,
                    'f6': f6,
                    'f_i_1': f_i_1,
                    'f_i_2': f_i_2,
                    'f_i_3': f_i_3,
                    # 'Expected_Life': Expected_Life,
                    'Time': elapsed_time
                })
                        
                current_iteration += 1

# ================== RESULTS REVIEW

df_total_f_results = pd.DataFrame(results)

# ================== df_info_STK

# print(f"df_info_STK:\n{df_info_STK.head()}\n")

print(f"📌 End Time\t: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

script_dir: c:\Users\hgvillanueva\OneDrive - Universidad Pontificia Comillas\Github\BESS-for-ancillary-services

📌 Start Time	: 2026-05-06 12:41:25

################################################################################
 *************** INFORMATION ABOUT SUITABLE ITERATIONS *************** 
Num. BatteryNodes: 1
Num. P_BESS: 1
Num. E_BESS: 1
 - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - -  - - 
Total Iterations: 1
Hours: 4657 to 4824
################################################################################


 ------------- BESS CASE: 500 kW, 2000 kWh ------------- 
--------------------------------------------------------------------------------
Iteration 1/1
🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹
Start Time	: 2026-05-06 12:41:25
CB_Monitored: Line.CB_202
h: 4656
h: 4657
h: 4658
h: 4659
h: 4660
h: 4661
h: 4662
h: 4663
h: 466

In [2]:
import pandas as pd

# Configurar pandas para mostrar todas las filas y columnas sin cortes
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

df_info_STK['Col_SoC'] = df_info_STK['Col_SoC'] * 100

print(f"df_info_STK: \n{df_info_STK}")

df_info_STK: 
        hr Col_SoC  P_BESS_t name_policy P_3f_cb_302  P_3f_cb_302_BESS_0  DSS_Total_Looses  switch_FR  P_bid_h  switch_PS  PS_peak  SW_FR_SoCmin  SW_FR_SoCmax  DSS_Total_Looses_BESS    f4_t     ISV     ICV  m_p_I2_I1  m_p_I2_I1_BESS  m_p_I0_I1  m_p_I0_I1_BESS  m_p_Normal  m_p_Normal_BESS    f3_t  f_i_1_t  f_i_2_t  f_i_3_t
46561    0 12.9343  228.4868           4    491.2199            262.7355         5368.5218     1.0000 238.1220     0.0000 262.7355        0.0000        0.0000              6484.3960 -0.0404 -0.0005  0.0004     9.6879          8.2869     9.6630          8.2519      1.0246           1.1508  0.2079  -0.1446  -0.1460   0.1232
46562    0 13.6944  168.9030           4    431.6307            262.7331         5368.0935     1.0000 238.1220     0.0000 262.7331        0.0000        0.0000              6152.9022 -0.0308 -0.0004  0.0003     9.6877          8.3300     9.6629          8.2966      1.0245           1.1178  0.1462  -0.1401  -0.1414   0.0910
46563    0 12.

In [3]:
df_info_STK.loc[df_info_STK.index[0] - 1] = df_info_STK.iloc[0]
df_info_STK = df_info_STK.sort_index()
df_info_STK.at[df_info_STK.index[0], 'Col_SoC'] = df_info_STK.at[df_info_STK.index[1], 'Col_SoC']

r_df_info_STK = df_info_STK.copy()

# Hora en la que quieres empezar a modificar los valores
# e_hora_f = 168
# Buscar las filas que coinciden con el valor de 'hr' igual a e_hora_f
matching_rows = df_info_STK[df_info_STK['hr'] == e_hora_f]

# Verificar si se encontró alguna fila con el valor de e_hora_f
if not matching_rows.empty:
    # Si se encuentra el índice, obtener el primer índice
    start_idx = matching_rows.index[0]
    print(f"start_idx: {start_idx}")
    
    # Modificar las columnas a partir de este índice
    df_info_STK.loc[start_idx:, 'name_policy'] = 0  # P_3f_cb_302_BESS_0
    df_info_STK.loc[start_idx:, 'P_3f_cb_302_BESS_0'] = None
    print(f"df_info_STK:\n{df_info_STK}\n")

print(df_info_STK.head())  

       hr Col_SoC  P_BESS_t name_policy P_3f_cb_302  P_3f_cb_302_BESS_0  DSS_Total_Looses  switch_FR  P_bid_h  switch_PS  PS_peak  SW_FR_SoCmin  SW_FR_SoCmax  DSS_Total_Looses_BESS    f4_t     ISV     ICV  m_p_I2_I1  m_p_I2_I1_BESS  m_p_I0_I1  m_p_I0_I1_BESS  m_p_Normal  m_p_Normal_BESS    f3_t  f_i_1_t  f_i_2_t  f_i_3_t
46560   0 12.9343  228.4868           4    491.2199            262.7355         5368.5218     1.0000 238.1220     0.0000 262.7355        0.0000        0.0000              6484.3960 -0.0404 -0.0005  0.0004     9.6879          8.2869     9.6630          8.2519      1.0246           1.1508  0.2079  -0.1446  -0.1460   0.1232
46561   0 12.9343  228.4868           4    491.2199            262.7355         5368.5218     1.0000 238.1220     0.0000 262.7355        0.0000        0.0000              6484.3960 -0.0404 -0.0005  0.0004     9.6879          8.2869     9.6630          8.2519      1.0246           1.1508  0.2079  -0.1446  -0.1460   0.1232
46562   0 13.6944  168.9030    

In [4]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import itertools

# Configurar la fuente del texto
mpl.rc('font', family='serif', serif='cmr10', size=12)
plt.rcParams['axes.unicode_minus'] = False

# Configurar la fuente de MathText para que use Computer Modern
mpl.rcParams['mathtext.fontset'] = 'cm'  # Usa las fuentes de Computer Modern para las ecuaciones
mpl.rcParams['mathtext.rm'] = 'serif'    # Usa 'serif' para el modo matemático normal
mpl.rcParams['mathtext.it'] = 'serif:italic'  # Itálica para símbolos matemáticos
mpl.rcParams['mathtext.bf'] = 'serif:bold'    # Negrita para símbolos matemáticos

# Configurar la fuente del texto
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
mpl.rc('font', family='serif', serif='cmr10', size=12)
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
###########################################################################################################
######################################### CREACION DE FIGURA

import os
from matplotlib.colors import to_rgba

# Obtener la ruta del directorio del script
# script_dir = os.getcwd()


# Crear la carpeta 'Outfiles_STK' en la misma ubicación que el script si no existe
output_dir = os.path.join(script_dir, 'Images')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"NEW output_dir: {output_dir}")

# Extraer los valores de las columnas 'P_3f_cb_302' y 'Col_SoC'
P_3f_cb_302 = df_info_STK['P_3f_cb_302']
Col_SoC = df_info_STK['Col_SoC']
###########################################################################################################
# >>>>>>>>>>>>>>>>>>>>>>>> GENERADOR DE GRÁFICOS

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
from matplotlib.colors import to_rgba

e_grid_linewidth_s = 0.25

def colored_line(x, y, colors, ax, **kwargs):
    for i in range(len(x) - 1):
        ax.plot(x[i:i+2], y[i:i+2], color=colors[i+1], **kwargs)
        # Use 'o' for a circular marker and set facecolors to 'none' for no fill
        ax.scatter(x[i], y[i], color=colors[i], marker='o', facecolors='none', **kwargs, s=5)

fig, axs = plt.subplots(3, 1, figsize=(30, 20), dpi=700, gridspec_kw={'height_ratios': [1, 1, 1], 'hspace': 0.30})

colors_dict = {
    1: 'yellowgreen',     # Daily BESS Charging
    2: 'tomato',           # BESS Idle
    3: 'darkorange',      # PS
    4: 'royalblue',       # FR
    5: 'lightgreen',         # BESS SoC Objective Charging
    0: to_rgba('white', alpha=0)  # Transparent (white with alpha 0)
}

colors_policy = df_info_STK['name_policy'].map(colors_dict).fillna('blue').to_numpy()

# Dibujar las líneas horizontales primero (detrás de todo)
axs[0].axhline(y=P_BESS, color='#D3D3D3', linestyle='--', linewidth=0.5, zorder=1)
axs[0].axhline(y=-P_BESS, color='#D3D3D3', linestyle='--', linewidth=0.5, zorder=1)
axs[0].axhline(y=0, color='#D3D3D3', linestyle='--', linewidth=0.5, zorder=1)

# Crear los ticks en el eje X en los múltiplos de 120
xticks = [x for x in range(df_info_STK.index.min(), df_info_STK.index.max() + 1) if x % 120 == 0]
xticklabels = [str((x // 10) - primera_hora + 1) for x in xticks]

# Graficar P_BESS_t en el primer eje con colores según la política
x = df_info_STK.index.to_numpy()
y = df_info_STK['P_BESS_t'].to_numpy()
colors = [colors_dict[val] for val in df_info_STK['name_policy']]

# Convertir el gráfico de líneas a barras y establecer zorder para que se impriman al final
axs[0].bar(x - 1.0, y, color=colors, width=1.2, zorder=2, align='edge')

# e_fontsize = 23
# e_fontsize_cientific = 23
# e_fontsize_cientific_zoom = 23
# e_fontsize_legend = 23
# e_fontsize_marks = 18

e_fontsize = 35
e_fontsize_cientific = 35
e_fontsize_cientific_zoom = 35
e_fontsize_legend = 35
e_fontsize_marks = 25
e_markersize = 15

# Configurar títulos y etiquetas
axs[0].set_xlabel('Time [h]', fontsize=e_fontsize)
axs[0].set_ylabel('BESS Power [kW]', fontsize=e_fontsize, labelpad=10)

# Configurar los límites y ticks para el eje X
axs[0].set_xlim([df_info_STK.index.min(), df_info_STK.index.max()])  # Añadir un pequeño margen en los extremos
axs[0].set_xticks(xticks)
axs[0].set_xticklabels(xticklabels, fontsize=e_fontsize)

# Configurar ticks y límites para el eje Y
axs[0].set_yticks([P_BESS, 0, -P_BESS])  # Solo los ticks de interés
axs[0].set_ylim(-P_BESS - 300, P_BESS + 300)  # Limitar el eje Y
axs[0].set_yticklabels([f"{int(tick)}" for tick in [P_BESS, 0, -P_BESS]], fontsize=e_fontsize)

# Configurar las líneas de cuadrícula y ticks para estar al frente
axs[0].grid(True, color='#D3D3D3', linestyle='--', linewidth=e_grid_linewidth_s, zorder=3)
axs[0].tick_params(axis='x', direction='out', length=6, width=1, colors='black', zorder=4) #, top=True)
axs[0].tick_params(axis='y', direction='out', length=6, width=1, colors='black', zorder=4) #, right=True)

# Configurar notación científica para el eje Y
axs[0].yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
axs[0].ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
axs[0].yaxis.get_offset_text().set_fontsize(e_fontsize_cientific)

# legend_1 = mlines.Line2D([], [], color='skyblue', label='Mode: BESS Charge ', linestyle='-', marker='o', markersize=5)
# legend_2 = mlines.Line2D([], [], color='royalblue', label='Mode: BESS Idle', linestyle='-', marker='o', markersize=5)
# legend_3 = mlines.Line2D([], [], color='blue', label='Mode: Service PS', linestyle='-', marker='o', markersize=5)
# legend_4 = mlines.Line2D([], [], color='blue', label='Mode: Service FRM', linestyle='-', marker='o', markersize=5)
# legend_5 = mlines.Line2D([], [], color='darkgray', label='Mode: Recovering SoC Objective', linestyle='-', marker='o', markersize=5)

# Agregar a la leyenda
# axs[0].legend(handles=[legend_1, legend_2, legend_3, legend_4, legend_5], prop={'size': 8}, ncol=5)
# axs[0].legend(handles=[legend_1, legend_3], prop={'size': 8}, ncol=3)

# Graficar Col_SoC en el segundo eje con colores según la política

y = df_info_STK['Col_SoC'].to_numpy()
colored_line(x, y, colors_policy, axs[1], linewidth= 0.75, zorder=4)
#axs[1].set_title(f'BESS SoC - Service: FR - BESS Properties: {P_BESS} kW {E_BESS} kWh - Time Period: {primera_hora} to {ultima_hora} hours', fontsize=15, fontname='Times New Roman')

axs[1].grid(True, color='#D3D3D3', linestyle='--', linewidth=e_grid_linewidth_s, zorder=3)

# Añadir líneas ligeras en los ticks de los ejes X e Y, incluyendo el límite superior
# axs[1].tick_params(axis='x', direction='out', length=6, width=1, colors='black', grid_color='gray', grid_alpha=0.8)
# axs[1].tick_params(axis='y', direction='out', length=6, width=1, colors='black', grid_color='gray', grid_alpha=0.8)

axs[1].set_xlabel('Time [h]', fontsize=e_fontsize) #, fontname='Times New Roman')
axs[1].set_ylabel('SOC [\%]', fontsize=e_fontsize, labelpad=25) #, fontname='Times New Roman')  #############################################################################
# axs[1].tick_params(axis='x', color='#D3D3D3', labelsize=12, rotation=0)
# axs[1].tick_params(axis='y', color='#D3D3D3', labelsize=12)
# SoC_min = (E_BESS/2 - P_BESS)/(E_BESS)
axs[1].set_yticks([10, 50, 90]) 
axs[1].set_ylim(0, 100)
# axs[1].grid(True)
axs[1].set_xlim([df_info_STK.index.min(), df_info_STK.index.max()])
axs[1].set_xticks(xticks)
axs[1].set_xticklabels(xticklabels, fontsize=e_fontsize, zorder=1)  #, fontname='Times New Roman')
axs[1].set_yticks(axs[1].get_yticks())
axs[1].set_yticklabels([f"{tick:.1f}" for tick in axs[1].get_yticks()], fontsize=e_fontsize, zorder=1)  #, fontname='Times New Roman')
axs[1].yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:.0f}'))

# Agregar líneas horizontales en y = 0.05 y y = 0.95
# axs[1].axhline(y=10, color='#D3D3D3', linestyle='--', linewidth=4, zorder=1)

linea_soc_objective = axs[1].axhline(y=50, color='gray', linestyle=':', linewidth=3, label=r'$\text{SOC}_{\text{obj}}$')
# legend_1 = mlines.Line2D([], [], color='skyblue', label='Mode: BESS Charge ', linestyle='-', marker='o', markersize=5)
# legend_2 = mlines.Line2D([], [], color='royalblue', label='Mode: BESS Idle', linestyle='-', marker='o', markersize=5)
# legend_3 = mlines.Line2D([], [], color='blue', label='Mode: Service PS', linestyle='-', marker='o', markersize=5)
# legend_4 = mlines.Line2D([], [], color='blue', label='Mode: Service FRM', linestyle='-', marker='o', markersize=5)
# legend_5 = mlines.Line2D([], [], color='darkgray', label='Mode: Recovering SoC Objective', linestyle='-', marker='o', markersize=5)

# axs[1].legend(handles=[legend_1, legend_2, legend_3, legend_4, legend_5, linea_soc_objective], prop={'size': 8}, ncol=6)
axs[1].legend(handles=[linea_soc_objective], prop={'size': e_fontsize-6}, ncol=1)

# Graficar P_3f_cb_302_BESS_0 y P_3f_cb_302 en el tercer eje con colores
y1 = df_info_STK['P_3f_cb_302_BESS_0'].to_numpy()
y2 = df_info_STK['P_3f_cb_302'].to_numpy()

# Cambiar el color de y1 a gris fuerte y y2 a otro color basado en la política
axs[2].plot(x, y1, color=to_rgba('black', alpha=0.5), linewidth=0.75, label='Without BESS', zorder=0)  # Línea gris con transparencia ajustada
colored_line(x, y2, colors_policy, axs[2], linewidth=0.75, alpha=0.8, zorder=0)  # Línea 2 con colores según la política y transparencia

axs[2].grid(True, color='#D3D3D3', linestyle='--', linewidth=e_grid_linewidth_s, zorder=5)

# Configurar título y etiquetas para el gráfico axs[2]
axs[2].set_xlabel('Time [h]', fontsize=e_fontsize)
axs[2].set_ylabel('CB_202 Power [kW]', fontsize=e_fontsize, labelpad=15) #############################################################################

# Configurar límites y ticks para el eje Y
axs[2].set_ylim(-250, 1750)  # <-- Ajustar el límite del eje Y de 0 a 1500
axs[2].set_yticks([i for i in range(0, 1501, 500)])  # <-- Fijar los ticks de 0 a 1500 con paso de 500

# Configurar etiquetas del eje Y
axs[2].set_yticklabels([f"{int(tick)}" for tick in axs[2].get_yticks()], fontsize=e_fontsize, zorder=1) 

# Configurar las líneas de cuadrícula
# axs[2].grid(True)

# Mantener la configuración original de los ticks y etiquetas en el eje X
axs[2].set_xlim([df_info_STK.index.min(), df_info_STK.index.max()])
axs[2].set_xticks(xticks)
axs[2].set_xticklabels(xticklabels, fontsize=e_fontsize, zorder=1) 

# Configurar notación científica para el eje Y
axs[2].yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
axs[2].ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
axs[2].yaxis.get_offset_text().set_fontsize(e_fontsize_cientific)

# p7 = axs[2].axhline(y=1500, color='black', linestyle='--', linewidth=0.75, label='Peak-demand threshold', alpha = 0.5 ,zorder=4)

#####

from matplotlib.legend_handler import HandlerTuple

l_linewidth = 5

# Crear los elementos para la leyenda, especificando solo la línea (sin el marcador)
p1, = axs[2].plot([1, 2.5, 3], color='yellowgreen', linewidth=l_linewidth, label="Line 1")  # Línea más gruesa
p2, = axs[2].plot([3, 2, 1], color='royalblue', linewidth=l_linewidth, label="Line 2")    # Línea más gruesa
p3, = axs[2].plot([3, 2, 1], color='darkorange', linewidth=l_linewidth, label="Line 3")   # Línea más gruesa
p4, = axs[2].plot([1, 2, 3], color='lightgreen', linewidth=l_linewidth, label="Line 4")   # Línea más gruesa
p5, = axs[2].plot([1, 2, 3], color='tomato', linewidth=l_linewidth, label="Line 5")      # Línea más gruesa
# Crear otra línea independiente para la segunda entrada en la leyenda
p6, = axs[2].plot([1, 2, 3], color='#666666', linewidth=3, label="Additional Line")  # Línea más gruesa

# Añadir ambas entradas a la leyenda
axs[2].legend(
    [(p2, p4, p5), p6],  # Primera entrada combina p1, p2, p3, p4, p5; Segunda entrada es p6
    ['With BESS', 'Without BESS'],  # Etiquetas para las entradas
    numpoints=1,
    handler_map={tuple: HandlerTuple(ndivide=None)},  # Evitar la división de la leyenda para el conjunto de líneas
    loc='lower right',  # Ubicación de la leyenda en la parte inferior derecha
    fontsize=e_fontsize-6,  # Tamaño de fuente
    ncol=3,
)

#####
# Ajustar el layout para eliminar los espacios adicionales
plt.tight_layout(pad=0.5)  # Reducir el padding entre los gráficos

# colors_dict = {
#     1: 'yellowgreen',     # Daily BESS Charging
#     2: 'tomato',           # BESS Idle
#     3: 'darkorange',      # PS
#     4: 'royalblue',       # FR
#     5: 'springreen'         # BESS SoC Objective Charging
# }

# Crear elementos para la leyenda general
# legend_1 = mlines.Line2D([], [], color='yellowgreen', label='Policy: Daily charging', linestyle='-', marker='o', markersize=5)
legend_2 = mlines.Line2D([], [], color='royalblue', label='Policy: Frequency regulation service', linestyle='-', marker='o', markersize=e_markersize)
# legend_3 = mlines.Line2D([], [], color='darkorange', label='Policy: Peak-shaving service', linestyle='-', marker='o', markersize=5)
legend_4 = mlines.Line2D([], [], color='lightgreen', label='Policy: Attaining SOC objective', linestyle='-', marker='o', markersize=e_markersize)
legend_5 = mlines.Line2D([], [], color='tomato', label='Policy: Idle', linestyle='-', marker='o', markersize=e_markersize)
# legend_5 = mlines.Line2D([], [], color='springreen', linestyle='--', label='SoC Objective')

# Agregar la leyenda general
#fig.legend(handles=[legend_1, legend_2, legend_3, legend_4, legend_5], loc='upper center', ncol=5, bbox_to_anchor=(0.5, 0.98), fontsize=10)
# fig.legend(handles=[legend_1, legend_2, legend_4, legend_5], loc='upper center', ncol=4, bbox_to_anchor=(0.5, 0.98), fontsize=10)
fig.legend(handles=[legend_2, legend_4, legend_5], loc='upper center', ncol=5, bbox_to_anchor=(0.5, 0.00), fontsize=e_fontsize_legend)
# fig.legend(handles=[legend_1, legend_2], loc='upper center', ncol=4, bbox_to_anchor=(0.5, 0.98), fontsize=10)

#############

# Ajustar el layout para evitar que la leyenda se superponga con los gráficos
plt.subplots_adjust(bottom=0.06)  # Ajustar la posición de la parte superior de la figura

# Agregar identificadores debajo de cada subgráfico con mayor separación
# axs[0].text(0.5, -0.41, 'a)', transform=axs[0].transAxes, fontsize=e_fontsize, fontweight='bold', va='center', ha='center')
# axs[1].text(0.5, -0.41, 'b)', transform=axs[1].transAxes, fontsize=e_fontsize, fontweight='bold', va='center', ha='center')
# axs[2].text(0.5, -0.41, 'c)', transform=axs[2].transAxes, fontsize=e_fontsize, fontweight='bold', va='center', ha='center')


#####################
# region
####

##########################
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
##########################

# Suponiendo que df_info_STK y colors_policy ya están definidos en tu entorno
x = df_info_STK.index.to_numpy()
y = df_info_STK['P_BESS_t'].to_numpy()

#-------------------------------------------

# Configuración de los parámetros de la lupa
e_linewith = 1.5
e_grid_linewidth = 0.5
zoom_1 = 650 #630
zoom_2 = 240 #300 #270
zoom_3 = -200

x_space = 1221 #1225 #1250 #1230
y_space = 290 #280 #250 #300 #100 #50 #10
zoom_ancho = 8000
zoom_alto = 350 

primera_hora_lupa_0 = 830 #910  # Ajusta el valor de acuerdo con tu caso
ultima_hora_lupa_0 = 851 #931   # Ajusta el valor de acuerdo con tu caso

#-------------------------------------------

# Limitar los datos de x e y para la lupa (ajustar el rango)
x_limited_0 = x[primera_hora_lupa_0:ultima_hora_lupa_0]  # Limitar los datos de x entre los índices
y_limited_0 = y[primera_hora_lupa_0:ultima_hora_lupa_0]  # Limitar los datos de y entre los índices
colors_policy_limited_0 = colors_policy[primera_hora_lupa_0:ultima_hora_lupa_0]  # Limitar también los colores para la lupa

# Crear la lupa en el gráfico (subplot 0)
axins_0 = inset_axes(axs[0], width="5%", height="30%", loc='upper left', bbox_to_anchor=(x_space, y_space + zoom_1 + 10, zoom_ancho, zoom_alto))

# Establecer los límites de la lupa (área a hacer zoom)
axins_0.set_xlim(min(x_limited_0), max(x_limited_0))
axins_0.set_ylim(min(y_limited_0), max(y_limited_0))  # Definir el rango Y según el zoom que deseas
axins_0.set_ylim(-P_BESS - 150, P_BESS + 150)  # Limitar el eje Y para el zoom de las barras

# Graficar las barras con los colores correspondientes en la lupa (subplot 0)
axins_0.bar(x_limited_0 - 1.0, y_limited_0, color=colors_policy_limited_0, width=1.2, zorder=2, align='edge')

# Establecer los ticks del eje Y para la lupa
yticks_lupa_0 = np.arange(-500, 501, 500)  # De -500 a 500 con paso de 500
axins_0.set_yticks(yticks_lupa_0)
axins_0.set_yticklabels([f"{tick}" for tick in yticks_lupa_0], fontsize=e_fontsize_marks)  # Ajuste de tamaño de la fuente

# Configuración de notación científica para el eje Y del primer gráfico
axins_0.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
axins_0.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
axins_0.yaxis.get_offset_text().set_fontsize(e_fontsize_marks)  # Para el texto del offset

# Establecer los ticks del eje X para la lupa (ajustar según el rango de x_limited_0)
xticks_lupa_0 = [x for x in range(min(x_limited_0), max(x_limited_0) + 1) if x % 10 == 0]  # Por ejemplo, cada 10 unidades
xticklabels_lupa_0 = [str((x // 10) - primera_hora + 1) for x in xticks_lupa_0]  # Ajuste de etiquetas para los ticks de X

axins_0.set_xticks(xticks_lupa_0)
axins_0.set_xticklabels(xticklabels_lupa_0, fontsize=e_fontsize_marks)  # Ajuste del tamaño de la fuente

# Indicar la zona de zoom en el gráfico principal
axins_0.grid(True, color='#D3D3D3', linestyle='--', linewidth=e_grid_linewidth, zorder=1)
axs[0].indicate_inset_zoom(axins_0, edgecolor="#000000", zorder=6)   # Ajustar el zorder al cuadro de zoom (mayor que las barras)
mark_inset(axs[0], axins_0, loc1=2, loc2=4, fc="none", ec="#94918f", linewidth=1)
# # # # # 

# # # # # 

x = df_info_STK.index.to_numpy()
y = df_info_STK['Col_SoC'].to_numpy()

primera_hora_lupa = primera_hora_lupa_0
ultima_hora_lupa = ultima_hora_lupa_0

# Limitar los datos de x e y para la lupa (ajustar el rango)
x_limited = x[primera_hora_lupa:ultima_hora_lupa]  # Limitar los datos de x entre los índices 300 y 450
y_limited = y[primera_hora_lupa:ultima_hora_lupa]  # Limitar los datos de y entre los índices 300 y 450
colors_policy_limited = colors_policy[primera_hora_lupa:ultima_hora_lupa]  # Limitar también los colores para la lupa

# Crear la lupa en el gráfico (subplot 1)
axins_1 = inset_axes(axs[1], width="5%", height="30%", loc='upper left', bbox_to_anchor=(x_space, y_space + zoom_2, zoom_ancho, zoom_alto))

# Establecer los límites de la lupa (área a hacer zoom)
axins_1.set_xlim(min(x_limited), max(x_limited))
axins_1.set_ylim(min(y_limited), max(y_limited))
axins_1.set_ylim(25, 55)  # Establecer un rango fijo de 0 a 1 para el eje Y de la lupa
yticks_lupa = np.arange(30, 51, 10)  # De 0.45 a 0.55 con paso de 0.01
axins_1.set_yticks(yticks_lupa)

# Graficar las líneas con los colores correspondientes en la lupa
colored_line(x_limited, y_limited, colors_policy_limited, axins_1, linewidth=e_linewith, zorder=4)

# Establecer los ticks del eje X para la lupa (ajustar según el rango de x_limited)
xticks_lupa = [x for x in range(min(x_limited), max(x_limited) + 1) if x % 10 == 0]  # Por ejemplo, cada 10 unidades
xticklabels_lupa = [str((x // 10) - primera_hora + 1) for x in xticks_lupa]

axins_1.set_xticks(xticks_lupa)
axins_1.set_xticklabels(xticklabels_lupa, fontsize=e_fontsize_marks)  # Ajusta el tamaño de la fuente

# Establecer los ticks del eje Y para la lupa (ajustar según el rango de y_limited)
yticks_lupa = [round(y_tick, 0) for y_tick in axins_1.get_yticks() if 0.0 <= y_tick <= 100]  # Ajustar al rango de 0 a 1
axins_1.set_yticks(yticks_lupa)
axins_1.set_yticklabels([f"{tick:.0f}" for tick in yticks_lupa], fontsize=e_fontsize_marks)  # Ajusta el tamaño de la fuente

axins_1.grid(True, color='#D3D3D3', linestyle='--', linewidth=e_grid_linewidth, zorder=1)

# Indicar la zona de zoom en el gráfico principal (esta línea dibuja el rectángulo de proyección)
axins_1.grid(True, color='#D3D3D3', linestyle='--', linewidth=e_grid_linewidth, zorder=1)
axs[1].indicate_inset_zoom(axins_1, edgecolor="#000000", zorder=6)   # Ajustar el zorder al cuadro de zoom (mayor que las barras)
mark_inset(axs[1], axins_1, loc1=2, loc2=4, fc="none", ec="#94918f", linewidth=1)
axins_1.axhline(y=50, color='gray', linestyle=':', linewidth=3, zorder=2, label=r'$\text{SOC}_{\text{obj}}$')
# # # # # 

# from mpl_toolkits.axes_grid1.inset_locator import inset_axes

x = df_info_STK.index.to_numpy()
y1 = df_info_STK['P_3f_cb_302_BESS_0'].to_numpy()
y2 = df_info_STK['P_3f_cb_302'].to_numpy()

primera_hora_lupa = primera_hora_lupa_0
ultima_hora_lupa = ultima_hora_lupa_0

# Limitar los datos de x e y para la lupa (ajustar el rango)
x_limited = x[primera_hora_lupa:ultima_hora_lupa]  # Limitar los datos de x entre los índices 300 y 450
y1_limited = y1[primera_hora_lupa:ultima_hora_lupa]  # Limitar los datos de y entre los índices 300 y 450
y_limited = y2[primera_hora_lupa:ultima_hora_lupa]  # Limitar los datos de y entre los índices 300 y 450

colors_policy_limited = colors_policy[primera_hora_lupa:ultima_hora_lupa]  # Limitar también los colores para la lupa

# Crear la lupa en el gráfico (subplot 1)
axins_2 = inset_axes(axs[2], width="5%", height="30%", loc='upper left', bbox_to_anchor=(x_space, y_space + zoom_3, zoom_ancho, zoom_alto))

# Establecer los límites de la lupa (área a hacer zoom)
axins_2.set_xlim(min(x_limited), max(x_limited))
axins_2.set_ylim(min(y_limited), max(y_limited))
axins_2.set_ylim(-50, 1550)  # Establecer un rango fijo de 0 a 1 para el eje Y de la lupa
yticks_lupa = np.arange(0, 1201, 500)  # De 0.45 a 0.55 con paso de 0.01
axins_2.set_yticks(yticks_lupa)

# Establecer los ticks del eje Y para la lupa
yticks_lupa_0 = np.arange(0, 1600, 750)  # De -500 a 500 con paso de 500
axins_2.set_yticks(yticks_lupa_0)
axins_2.set_yticklabels([f"{tick}" for tick in yticks_lupa_0], fontsize=e_fontsize_marks)  # Ajuste de tamaño de la fuente

# Configuración de notación científica para el eje Y del primer gráfico
axins_2.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
axins_2.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
axins_2.yaxis.get_offset_text().set_fontsize(e_fontsize_marks)  # Para el texto del offset

axins_2.plot(x_limited, y1_limited, color='#666666', linewidth=e_linewith, zorder=1)

# Graficar las líneas con los colores correspondientes en la lupa
colored_line(x_limited, y_limited, colors_policy_limited, axins_2, linewidth=e_linewith, zorder=4)

# Establecer los ticks del eje X para la lupa (ajustar según el rango de x_limited)
xticks_lupa = [x for x in range(min(x_limited), max(x_limited) + 1) if x % 10 == 0]  # Por ejemplo, cada 10 unidades
xticklabels_lupa = [str((x // 10) - primera_hora + 1) for x in xticks_lupa]

axins_2.set_xticks(xticks_lupa)
axins_2.set_xticklabels(xticklabels_lupa, fontsize=e_fontsize_marks)  # Ajusta el tamaño de la fuente

axins_2.grid(True, color='#D3D3D3', linestyle='--', linewidth=e_grid_linewidth, zorder=1)
axs[2].indicate_inset_zoom(axins_2, edgecolor="#000000", zorder=6)   # Ajustar el zorder al cuadro de zoom (mayor que las barras)
mark_inset(axs[2], axins_2, loc1=2, loc2=4, fc="none", ec="#94918f", linewidth=1)
# # # # # 


# # Crear las líneas de proyección para conectar las lupas entre sí
# mark_inset(axs[0], axins_0, loc1=2, loc2=4, fc="none", ec="#94918f")
# mark_inset(axs[1], axins_1, loc1=2, loc2=4, fc="none", ec="#94918f")
# mark_inset(axs[2], axins_2, loc1=2, loc2=4, fc="none", ec="#94918f")

#plt.title(f'Hour\t{e_hora_f}', pad=410, fontsize=e_fontsize + 3, fontweight='bold')  # Añadir título y ajustar su posición
# plt.title(f'Hour\t{e_hora_f}', pad=415, fontsize=e_fontsize + 5)
# plt.title(r'$\text{Hour}$' + f"\t{e_hora_f-1}", pad=410, fontsize=e_fontsize + 3)

# Engrosar los bordes de los subgráficos
e_grosor_marco_subplots = 2
for ax in axs:
    for spine in ax.spines.values():
        spine.set_linewidth(e_grosor_marco_subplots)  # Puedes ajustar el valor aquí para cambiar el grosor

######################
#endregion

now = datetime.now()
fecha_hora = now.strftime("%Y%m%d_%H%M%S")

# Crear la carpeta 'Outfiles_STK' en la misma ubicación que el script si no existe
output_dir = os.path.join(script_dir, 'Images')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Guardar la figura en la carpeta 'Outfiles_STK' sin mostrarla
# nombre_archivo = f'{fecha_hora}_FR_{P_BESS}kW_{E_BESS}kWh_{primera_hora}to{ultima_hora}.pdf'
nombre_archivo = f'{Svc}_Fig_09_Weekly_Operation_{P_BESS}kW_{E_BESS}kWh_{primera_hora}to{ultima_hora}_{fecha_hora}.pdf'
output_path = os.path.join(output_dir, nombre_archivo)
plt.savefig(output_path, bbox_inches='tight', dpi=300)
plt.show()
# plt.close()  # Cerrar la figura para no mostrarla

print(f"Gráfico guardado en {output_path}")

# Guardar el DataFrame en una hoja de Excel



NEW output_dir: c:\Users\hgvillanueva\OneDrive - Universidad Pontificia Comillas\Github\BESS-for-ancillary-services\Images


C:\Users\hgvillanueva\AppData\Local\Temp\ipykernel_23184\3310665634.py:218: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(pad=0.5)  # Reducir el padding entre los gráficos
